In [6]:
from google.colab import auth, drive
import os
from pathlib import Path

# 1. Mount Drive
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)

# 2. Define the path correctly
BASE_PATH = '/content/drive/MyDrive'

# REMOVE the leading slash here:
FOLDER_PATH = 'fire_detection/DataSet/Safe_fire/smoking'

IMAGE_DIR = os.path.join(BASE_PATH, FOLDER_PATH)

# 3. Check if it exists and count files
if os.path.isdir(IMAGE_DIR):
    total = sum(1 for _, _, fs in os.walk(IMAGE_DIR) for f in fs)
    print(f'Found {total} file(s) in {IMAGE_DIR}')
else:
    print(f'ERROR: Folder not found at {IMAGE_DIR}')
    print("Check for typos or case-sensitivity (e.g., 'DataSet' vs 'dataset').")

Mounted at /content/drive
Found 200 file(s) in /content/drive/MyDrive/fire_detection/DataSet/Safe_fire/smoking


In [7]:
!pip install -q "transformers==4.45.2" accelerate einops timm Pillow opencv-python-headless numpy torch torchvision
print('Installation Done.')

Installation Done.


In [8]:
import pathlib, sys
import torch
from transformers import AutoModel, AutoTokenizer

# --- STEP 3A: APPLY PATCH ---
CACHE_ROOT = pathlib.Path.home() / '.cache/huggingface/modules/transformers_modules/OpenGVLab'

BAD_LINE  = 'dpr = [x.item() for x in torch.linspace(0, config.drop_path_rate, config.num_hidden_layers)]'
GOOD_LINE = ('_n   = int(config.num_hidden_layers)\n'
             '        _end = config.drop_path_rate\n'
             '        _end = _end if isinstance(_end, float) else 0.1\n'
             '        dpr  = [_end * i / max(_n - 1, 1) for i in range(_n)]')

for vit_file in CACHE_ROOT.rglob('modeling_intern_vit.py'):
    code = vit_file.read_text(encoding='utf-8')
    if BAD_LINE in code:
        vit_file.write_text(code.replace(BAD_LINE, GOOD_LINE), encoding='utf-8')
        print(f'Patched file: {vit_file}')

# Clear existing modules to ensure patch applies
evicted = [k for k in list(sys.modules) if 'internvl' in k.lower() or 'intern_vit' in k.lower()]
for k in evicted:
    del sys.modules[k]

# --- STEP 3B: LOAD MODEL ---
MODEL_ID = 'OpenGVLab/InternVL2_5-4B'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE    = torch.bfloat16 if DEVICE == 'cuda' else torch.float32

print(f'Loading {MODEL_ID} on {DEVICE.upper()}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
model     = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=False,
).eval().to(DEVICE)

print(f'Model ready | device: {DEVICE.upper()} | dtype: {DTYPE}')

Loading OpenGVLab/InternVL2_5-4B on CUDA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.43G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Model ready | device: CUDA | dtype: torch.bfloat16


In [9]:
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode

MAX_NEW_TOKENS = 40
IMAGE_SIZE     = 448
MAX_TILES      = 6
IMAGE_EXTS     = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
VIDEO_EXTS     = {'.mp4', '.avi', '.mov', '.mkv', '.wmv', '.m4v'}

# The final prompt extracted from your notebook
PROMPT = (
    "Write a concise, 10-word description of this scene focusing on the source of combustion and resulting emissions. "
    "Clearly identify the specific object being held and the nature of any visible smoke or vapor. "
    "Be highly specific about the person's interaction with the object."
)

STOPWORDS = {'a','an','the','is','are','was','were','be','been','being','in','on','at','to','of','for','with','by','from','into','and','or','but','as','its','it','this','that','there','their','they','has','have','had','do','does','did','i','we','you','he','she','over','under','near','between','through','around','against','during','without','within'}

def build_transform(size=IMAGE_SIZE):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB')),
        T.Resize((size, size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])

def dynamic_preprocess(image, min_num=1, max_num=MAX_TILES, image_size=IMAGE_SIZE):
    w, h = image.size
    aspect = w / h
    ratios = sorted({(i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if min_num <= i * j <= max_num}, key=lambda x: x[0] * x[1])
    best = min(ratios, key=lambda r: abs(aspect - r[0] / r[1]))
    tw, th = image_size * best[0], image_size * best[1]
    resized = image.resize((tw, th))
    tiles = [resized.crop(((idx % best[0]) * image_size, (idx // best[0]) * image_size, ((idx % best[0]) + 1) * image_size, ((idx // best[0]) + 1) * image_size)) for idx in range(best[0] * best[1])]
    tiles.append(image.resize((image_size, image_size)))
    return tiles

def get_phrase(image):
    pixel_values = torch.stack([build_transform()(t) for t in dynamic_preprocess(image.convert('RGB'))]).to(DTYPE).to(DEVICE)
    return model.chat(tokenizer, pixel_values, PROMPT, dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)).strip()

def analyse_phrase(phrase):
    words = phrase.split()
    tokens = [w.strip('.,;:!?\'"').lower() for w in words if w.strip('.,;:!?\'"').lower() not in STOPWORDS and w.strip('.,;:!?\'"')]
    return {'word_count': len(words), 'meaningful_tokens': tokens, 'token_count': len(tokens)}

In [10]:
import csv

root = Path(IMAGE_DIR)
out_path = Path('/content/drive/MyDrive/fire_detection/internvl_results_smoking.csv')
all_files = sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS | VIDEO_EXTS)

print(f'Processing {len(all_files)} files...\n' + '─' * 60)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, 'w', encoding='utf-8', newline='') as out:
    writer = csv.writer(out)
    writer.writerow(['filename', 'category_folder', 'phrase', 'meaningful_tokens', 'word_count', 'meaningful_token_count'])

    for idx, path in enumerate(all_files, 1):
        rel = path.relative_to(root)
        category = rel.parent.name if rel.parent != Path('.') else root.name
        print(f'[{idx}/{len(all_files)}] {rel}')

        try:
            phrase = get_phrase(Image.open(path))
            info = analyse_phrase(phrase)
            writer.writerow([rel.name, category, phrase, ' '.join(info['meaningful_tokens']), info['word_count'], info['token_count']])
            out.flush()
        except Exception as e:
            print(f'  ERROR: {e}')
            writer.writerow([rel.name, category, f'ERROR: {e}', '', '', ''])

print(f'\nDone! Results saved to {out_path}')

Processing 200 files...
────────────────────────────────────────────────────────────
[1/200] smoking1.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


[2/200] smoking10.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


[3/200] smoking100.jpg


...

[199/200] smoking98.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


[200/200] smoking99.jpg


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



Done! Results saved to /content/drive/MyDrive/fire_detection/internvl_results_smoking.csv


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
